# Calculus and Optimization for ML
### Interactive Notebook for AI/ML Interview Preparation

Covers gradients, gradient descent variants, learning rates, convexity, and optimizers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Numerical Gradient

In [ ]:
def f(x): return x**2 - 4*x + 4  # f(x) = (x-2)^2, minimum at x=2
def numerical_gradient(f, x, h=1e-5):
    return (f(x+h) - f(x-h)) / (2*h)

x_vals = np.linspace(-1, 5, 100)
grads = [numerical_gradient(f, x) for x in x_vals]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vals, f(x_vals), 'b-', label='f(x) = (x-2)²')
ax.plot(x_vals, grads, 'r--', label="f'(x) = 2(x-2)")
ax.axhline(y=0, color='gray', linestyle=':')
ax.axvline(x=2, color='gray', linestyle=':', alpha=0.5)
ax.legend(); ax.set_title('Function and its Gradient')
plt.tight_layout(); plt.show()

---
## 2. Gradient Descent — 1D and 2D

In [ ]:
# 1D Gradient Descent
def grad_f(x): return 2*x - 4

x = 5.0  # starting point
lr = 0.1
history = [x]
for _ in range(20):
    x = x - lr * grad_f(x)
    history.append(x)

fig, ax = plt.subplots(figsize=(8, 4))
x_range = np.linspace(-1, 6, 100)
ax.plot(x_range, f(x_range), 'b-')
ax.plot(history, [f(h) for h in history], 'ro-', markersize=4, label='GD path')
ax.set_title(f'1D Gradient Descent: converged to x={history[-1]:.4f}')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 2D Gradient Descent on f(x,y) = x^2 + 3*y^2
def f2d(x, y): return x**2 + 3*y**2
def grad2d(x, y): return np.array([2*x, 6*y])

pos = np.array([4.0, 3.0])
lr = 0.1
path = [pos.copy()]
for _ in range(30):
    pos = pos - lr * grad2d(*pos)
    path.append(pos.copy())
path = np.array(path)

fig, ax = plt.subplots(figsize=(7, 6))
x_r = np.linspace(-5, 5, 100)
y_r = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x_r, y_r)
ax.contour(X, Y, f2d(X, Y), levels=20, cmap='viridis')
ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=3, linewidth=1)
ax.plot(0, 0, 'g*', markersize=15, label='Minimum')
ax.set_title('2D Gradient Descent'); ax.legend()
plt.tight_layout(); plt.show()

---
## 3. Learning Rate Effect

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, lr, title in zip(axes, [0.01, 0.1, 0.95],
                        ['Too Small (0.01)', 'Just Right (0.1)', 'Too Large (0.95)']):
    x = 5.0
    hist = [x]
    for _ in range(30):
        x = x - lr * grad_f(x)
        hist.append(x)
    x_r = np.linspace(-2, 8, 100)
    ax.plot(x_r, f(x_r), 'b-')
    ax.plot(hist, [f(h) for h in hist], 'ro-', markersize=3)
    ax.set_title(f'{title}\nFinal x={hist[-1]:.3f}')
    ax.set_ylim(-1, 20)
plt.tight_layout(); plt.show()

---
## 4. SGD vs Momentum vs Adam

In [ ]:
# Compare optimizers on f(x,y) = x^2 + 10*y^2 (elongated valley)
def f_valley(pos): return pos[0]**2 + 10*pos[1]**2
def grad_valley(pos): return np.array([2*pos[0], 20*pos[1]])

def run_optimizer(name, start, n_steps=50, lr=0.05):
    pos = start.copy()
    path = [pos.copy()]
    v = np.zeros(2)  # momentum/Adam state
    m = np.zeros(2)  # Adam state
    for t in range(1, n_steps+1):
        g = grad_valley(pos)
        if name == 'SGD':
            pos = pos - lr * g
        elif name == 'Momentum':
            v = 0.9 * v + g
            pos = pos - lr * v
        elif name == 'Adam':
            m = 0.9*m + 0.1*g
            v = 0.999*v + 0.001*g**2
            m_hat = m / (1 - 0.9**t)
            v_hat = v / (1 - 0.999**t)
            pos = pos - lr * m_hat / (np.sqrt(v_hat) + 1e-8)
        path.append(pos.copy())
    return np.array(path)

start = np.array([4.0, 3.0])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['SGD', 'Momentum', 'Adam']):
    path = run_optimizer(name, start)
    x_r = np.linspace(-5, 5, 100); y_r = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x_r, y_r)
    ax.contour(X, Y, X**2 + 10*Y**2, levels=20, cmap='viridis')
    ax.plot(path[:,0], path[:,1], 'ro-', markersize=2, linewidth=1)
    ax.set_title(f'{name} ({len(path)-1} steps)'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

---
## 5. Convex vs Non-Convex

In [ ]:
x = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, x**2, 'b-', linewidth=2)
axes[0].set_title('Convex: f(x) = x² — one global minimum')
axes[1].plot(x, x**4 - 3*x**2 + 1, 'r-', linewidth=2)
axes[1].set_title('Non-Convex: f(x) = x⁴-3x²+1 — local minima!')
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()
print('Convex → guaranteed global minimum with GD')
print('Non-convex → GD may get stuck in local minima (most neural nets are non-convex!)')

---
## Key Interview Takeaways

1. **Gradient** points in direction of steepest ascent; we go opposite for minimization
2. **Learning rate** is the most critical hyperparameter — too high diverges, too low stalls
3. **SGD** adds noise which helps escape local minima but is noisy
4. **Adam** combines momentum + adaptive learning rates — most popular default
5. **Convex** functions guarantee global optimum; neural nets are non-convex but still trainable